# Ensure cleaned silver tables are used for analysis

In [0]:
USE CATALOG global_stock_market;
USE SCHEMA silver;

SELECT current_catalog() , current_schema();

In [0]:
DROP SCHEMA IF EXISTS  global_stock_market.gold CASCADE;
CREATE SCHEMA IF NOT EXISTS global_stock_market.gold;

# Key Business Questions

## 1. How have major global stock indices performed over the last 5 years?  

In [0]:
CREATE OR REPLACE TABLE global_stock_market.gold.annual_returns AS

WITH latest_year AS (
    SELECT YEAR(MAX(Date)) AS max_year -- to get 2026
    FROM index_levels
),

yearly_prices AS (
    SELECT
        `Index`,
        YEAR(Date) AS Year,
        Date,
        Close,
        Country,

        ROW_NUMBER() OVER (
            PARTITION BY `Index`, YEAR(Date)
            ORDER BY Date DESC -- ensure the last date will be at the first row/ ranked 1
        ) AS last_day

    FROM index_levels
),

year_end_prices AS (
    SELECT
        yp.`Index`,
        yp.Year,
        yp.Date AS Ending_Date,
        yp.Close AS Ending_Close,
        yp.Country

    FROM yearly_prices yp
    CROSS JOIN latest_year ly

    WHERE yp.last_day = 1
      AND yp.Year >= ly.max_year - 5
      AND yp.Year <= ly.max_year
)

SELECT
    curr.`Index`,
    curr.Year,
    prev.Ending_Close AS Starting_Close,
    curr.Ending_Close,
    ROUND(
        (
            curr.Ending_Close / prev.Ending_Close - 1
        ) * 100,
        2
    ) AS Annual_Return_Percentage,
    curr.Country

FROM year_end_prices curr

JOIN year_end_prices prev  -- self join to get the previous year
    ON curr.`Index` = prev.`Index`
    AND curr.Year = prev.Year + 1 

ORDER BY
    curr.`Index`,
    curr.Year;



In [0]:
SELECT Index, MAX(date)
from index_levels
WHERE YEAR(Date) = 2026
GROUP BY Index

In [0]:
SELECT * FROM global_stock_market.gold.annual_returns

## 2. Which indices generated the highest year‑to‑date (YTD) returns?  (2025-2026)

In [0]:
CREATE OR REPLACE TABLE global_stock_market.gold.ytd_returns AS

SELECT
    curr.Index,
    curr.Country,
    prev.Ending_Close AS Close_2025,
    curr.Ending_Close AS Close_2026,
    ROUND(
        (curr.Ending_Close / prev.Ending_Close - 1) * 100,
        2
    ) AS YTD_Return_Percentage
FROM global_stock_market.gold.annual_returns AS curr
JOIN global_stock_market.gold.annual_returns AS prev  -- self join to get the previous year
ON curr.Index = prev.Index
WHERE curr.Year = 2026
AND prev.Year = 2025
ORDER BY YTD_Return_Percentage DESC;



In [0]:
SELECT * FROM global_stock_market.gold.ytd_returns

## 3. How do trends differ across months and years?  

### 3.1 Month by Month returns for each index

In [0]:
CREATE OR REPLACE TABLE global_stock_market.gold.monthly_index_trends AS

WITH ranked_prices AS (
    SELECT
        `Index`,
        Country,
        Date,
        Close,
        -- Extract the year and month for monthly grouping
        YEAR(Date) AS Year,
        MONTH(Date) AS Month_Number,

        -- Rank trading dates from earliest to latest within each index and month
        -- Rank 1 identifies the first available trading day of the month
        ROW_NUMBER() OVER (
            PARTITION BY `Index`, YEAR(Date), MONTH(Date)
            ORDER BY Date ASC
        ) AS first_day_rank,

        -- Rank trading dates from latest to earliest within each index and month
        -- Rank 1 identifies the last available trading day of the month
        ROW_NUMBER() OVER (
            PARTITION BY `Index`, YEAR(Date), MONTH(Date)
            ORDER BY Date DESC
        ) AS last_day_rank

    FROM global_stock_market.silver.index_levels
    WHERE Close IS NOT NULL
),

monthly_prices AS (
    SELECT
        `Index`,
        Country,
        Year,
        Month_Number,

        -- Retrieve the closing price on the first available trading day
        MAX(CASE
            WHEN first_day_rank = 1 THEN Close
        END) AS First_Close,

        -- Retrieve the closing price on the last available trading day
        MAX(CASE
            WHEN last_day_rank = 1 THEN Close
        END) AS Month_End_Close,

        -- Retrieve the date of the last available trading day
        MAX(CASE
            WHEN last_day_rank = 1 THEN Date
        END) AS Month_End_Date

    FROM ranked_prices
    GROUP BY
        `Index`,
        Country,
        Year,
        Month_Number
),

previous_month AS (
    SELECT
        *,
        -- Retrieve the previous available month's closing price
        -- for the same index, ordered chronologically
        LAG(Month_End_Close) OVER (
            PARTITION BY `Index`
            ORDER BY Year, Month_Number
        ) AS Previous_Month_End_Close

    FROM monthly_prices
)

SELECT
    `Index`,
    Country,
    Year,
    Month_Number,
    Month_End_Date,
    First_Close,
    Month_End_Close,
    Previous_Month_End_Close,
    
    -- Calculate monthly return using consecutive month-end closing prices
    -- Formula: (Current month-end close / Previous month-end close - 1) * 100
    ROUND(
        (Month_End_Close / Previous_Month_End_Close - 1) * 100,
        2
    ) AS Monthly_Return_Percentage

FROM previous_month;


In [0]:
SELECT * FROM global_stock_market.gold.monthly_index_trends

### 3.2 Year by year returns for each index

In [0]:
CREATE OR REPLACE TABLE global_stock_market.gold.yearly_index_trends AS

WITH yearly_prices AS (
    SELECT
        `Index`,
        Country,
        YEAR(Date) AS Year,
        Date,
        Close,

        ROW_NUMBER() OVER (
            PARTITION BY `Index`, YEAR(Date)
            ORDER BY Date DESC
        ) AS last_day_rank

    FROM global_stock_market.silver.index_levels
    WHERE Close IS NOT NULL
),

year_end_prices AS (
    SELECT
        `Index`,
        Country,
        Year,
        Date AS Year_End_Date,
        Close AS Year_End_Close

    FROM yearly_prices
    WHERE last_day_rank = 1
),

previous_year AS (
    SELECT
        *,
        LAG(Year_End_Close) OVER (
            PARTITION BY `Index`
            ORDER BY Year
        ) AS Previous_Year_End_Close

    FROM year_end_prices
)

SELECT
    `Index`,
    Country,
    Year,
    Year_End_Date,
    Year_End_Close,
    Previous_Year_End_Close,

    ROUND(
        (Year_End_Close / Previous_Year_End_Close - 1) * 100,
        2
    ) AS Annual_Return_Percentage

FROM previous_year;


In [0]:
SELECT * FROM global_stock_market.gold.yearly_index_trends

## 4. Volatility and Risk

### 4.1 Which indices and country exhibit the highest volatility? (more price fluctuates, more volaties , hence more risky)

In [0]:
CREATE OR REPLACE TABLE global_stock_market.gold.index_volatility_stats AS

WITH volatility_stats AS (
    SELECT
        Index,
        Country,
        STDDEV(Daily_Returns_Percentage) AS Daily_volatility,
        AVG(Daily_Returns_Percentage)AS Average_Daily_Return,
        COUNT(Daily_Returns_Percentage) AS Number_Of_Trading_Days,
        MIN(Date) AS Start_Date,
        MAX(Date) AS End_Date
    FROM global_stock_market.silver.index_levels
    WHERE Daily_Returns_Percentage IS NOT NULL
    GROUP BY Index, Country
)

SELECT *
FROM volatility_stats


In [0]:
CREATE OR REPLACE TABLE global_stock_market.gold.daily_volatility_stats AS

SELECT 
    Index,
    Country,
    Number_Of_Trading_Days,
    Start_Date,
    End_Date,
    ROUND(Average_Daily_Return, 4) AS Average_Daily_Return,
    ROUND(Daily_volatility, 4) AS Daily_volatility,

    CASE
        WHEN Daily_volatility < 1 THEN "Low"
        WHEN Daily_volatility BETWEEN 1 AND 1.5 THEN "Medium"
        WHEN Daily_volatility >= 1.6 THEN "High"
    END AS Daily_Volatility_Category
FROM 
    global_stock_market.gold.index_volatility_stats
ORDER BY 
    daily_volatility DESC

In [0]:
SELECT * FROM global_stock_market.gold.daily_volatility_stats

### 4.1.1 Yearly volatility

In [0]:
CREATE OR REPLACE TABLE global_stock_market.gold.yearly_volatility_stats AS

SELECT
    Index,
    Country,
    Region,
    YEAR(Date) AS Year,
    COUNT(Daily_Returns_Percentage) AS Trading_Days,
    ROUND(STDDEV_SAMP(Daily_Returns_Percentage), 4) AS Annual_Volatility_Percentage
FROM global_stock_market.silver.index_levels
WHERE Daily_Returns_Percentage IS NOT NULL
GROUP BY
    Index,
    Country,
    Region,
    YEAR(Date)
ORDER BY
    Index,
    Year;


In [0]:
SELECT * FROM global_stock_market.gold.yearly_volatility_stats

## 5. Country & Regional Comparisons  

### 5.1 How do countries compare in average daily returns?  

In [0]:
CREATE OR REPLACE TABLE global_stock_market.gold.avg_daily_returns AS

SELECT
    Index,
    Country,
    ROUND(AVG(Daily_Returns_Percentage), 4)
        AS Average_Daily_Return,
    COUNT(Daily_Returns_Percentage) AS Trading_Days
FROM global_stock_market.silver.index_levels
WHERE Daily_Returns_Percentage IS NOT NULL
GROUP BY Index, Country
ORDER BY Average_Daily_Return DESC;


In [0]:
SELECT * FROM global_stock_market.gold.avg_daily_returns

### 5.2 Are there significant differences between regions?  

In [0]:
CREATE OR REPLACE TABLE global_stock_market.gold.region_daily_returns AS

SELECT
    Region,
    ROUND(AVG(Daily_Returns_Percentage), 4)
        AS Average_Daily_Return,
    COUNT(Daily_Returns_Percentage) AS Trading_Days
FROM global_stock_market.silver.index_levels
WHERE Daily_Returns_Percentage IS NOT NULL
GROUP BY Region
ORDER BY Average_Daily_Return DESC;


In [0]:
select * from global_stock_market.gold.region_daily_returns

### 5.3 Which countries show the strongest long‑term performance?  (Last 10 years performance)

In [0]:

CREATE OR REPLACE TABLE global_stock_market.gold.country_long_term_performance AS

WITH start_prices AS (

    -- Retrieve each index's year-end closing price for 2014
    -- This serves as the starting price for the 10-year comparison
    SELECT
        `Index`,
        Country,
        Year_End_Close AS Starting_Close,
        Year_End_Date AS Start_Date

    FROM global_stock_market.gold.yearly_index_trends

    WHERE Year = 2014
),

end_prices AS (

    -- Retrieve each index's year-end closing price for 2025
    -- This serves as the ending price for the comparison
    SELECT
        `Index`,
        Country,
        Year_End_Close AS Ending_Close,
        Year_End_Date AS End_Date

    FROM global_stock_market.gold.yearly_index_trends

    WHERE Year = 2025
),

performance AS (

    -- Match each index's starting price with its ending price
    -- so that performance is calculated for the same index and country
    SELECT
        s.`Index`,
        s.Country,
        s.Start_Date,
        e.End_Date,
        s.Starting_Close,
        e.Ending_Close

    FROM start_prices s

    INNER JOIN end_prices e
        ON s.`Index` = e.`Index`
        AND s.Country = e.Country

    -- Exclude records with zero or negative prices,
    -- which would make return calculations invalid
    WHERE s.Starting_Close > 0
      AND e.Ending_Close > 0
)

SELECT
    `Index`,
    Country,
    Start_Date,
    End_Date,
    Starting_Close,
    Ending_Close,

    -- Calculate the total percentage price change over the 10-year period
    -- Formula: (Ending Close / Starting Close - 1) * 100
    ROUND(
        (Ending_Close / Starting_Close - 1) * 100,
        2
    ) AS Cumulative_Return_Percentage,

    -- Calculate the compound annual growth rate (CAGR)
    -- This represents the equivalent annualized growth rate over 10 years
    -- Formula: ((Ending Close / Starting Close) ^ (1 / 11) - 1) * 100
    ROUND(
        (POWER(Ending_Close / Starting_Close, 1.0 / 11) - 1) * 100,
        2
    ) AS CAGR_Percentage

FROM performance

-- Rank the strongest annualized performers first
ORDER BY CAGR_Percentage DESC;


In [0]:
SELECT * FROM global_stock_market.gold.country_long_term_performance

# Key Findings & Insights

## 1. Turkey's BIST100 — Highest Returns, Highest Risk

Turkey's BIST100 dominates nearly every performance metric but also carries the greatest risk:

| Metric | BIST100 (Turkey) | Next Best |
|---|---|---|
| 10-yr CAGR | 31.7% | Brazil 14.03% |
| 10-yr cumulative return | +1,470% | Brazil +272% |
| YTD 2026 return | +23.73% | Brazil +18.25% |
| Avg daily return | 0.087% | Brazil 0.0518% |
| Daily volatility | 2.0022 (High) | Brazil 1.698 (High) |

Turkey's 10-year cumulative return of **+1,470%** is roughly **5.4x** the next best market. However, this comes with the highest daily volatility (2.0022), placing it in the only "High" risk tier alongside Brazil. In 2022 alone, BIST100 surged +196.56%.

## 2. China's SSE — Worst Long-Term Performer Despite Moderate Risk

China's SSE generated only **+12.14% cumulative** over 10 years (1.15% CAGR) — essentially flat. It ranks last in long-term performance, its YTD 2026 return is a modest +2.85%, and its average daily return (0.0274%) is near the bottom. Yet its volatility (1.4539) is "Medium" — investors took on meaningful risk for almost no reward.

## 3. US S&P 500 — Best Risk-Adjusted Return

The S&P 500 ranks **3rd in 10-year CAGR (12.85%)** but has the **2nd lowest volatility (1.2179, Medium)**. This delivers top-tier returns with comparatively low risk — the best risk-adjusted profile of any index. The UK's FTSE100, by contrast, has the lowest volatility (1.1292) but also the lowest average daily return (0.0136%), making it stable but low-reward.

## 4. Emerging Markets Outperform Developed Markets — But With Higher Risk

Regional average daily returns:

| Region | Avg Daily Return |
|---|---|
| Europe/Asia (Turkey) | 0.087% |
| South America (Brazil) | 0.0518% |
| Middle East (Saudi Arabia) | 0.0402% |
| Asia (Japan, China, India, Indonesia) | 0.0365% |
| North America (US) | 0.0311% |
| Europe (Germany, UK) | 0.0217% |

The top three regions (Turkey, Brazil, Saudi Arabia) are all emerging/frontier markets, and they also occupy the top two volatility spots. Developed European markets (Germany + UK) sit at the bottom with the lowest average daily returns.

## 5. No Index Qualifies as "Low" Volatility

Every single index falls into either "Medium" or "High" volatility — none below 1.0. Even the most stable (FTSE100 at 1.1292) still exhibits moderate daily swings. All 10 global indices experienced significant price fluctuations over their ~26-year histories.

## Why These Patterns Exist

* **Turkey's extreme outperformance** reflects high inflation dynamics — Turkish equities are priced in a currency that has depreciated massively, so nominal index levels inflate dramatically in lira terms. Returns are real in local currency but heavily eroded in USD terms. 
* **Emerging market risk premium**: Brazil and Turkey offer higher returns because they compensate investors for political instability, currency risk, and economic volatility.
* **China's underperformance** reflects regulatory crackdowns, trade tensions, and a structurally slowing economy over the 2015–2025 period.
* **The S&P 500's efficiency** reflects the depth, liquidity, and tech-driven growth of US equity markets, which attract global capital and tend to price in risk more efficiently.
* **Developed Europe's low returns** (Germany 8.59% CAGR, UK 4.75% CAGR) reflect mature economies with slower growth trajectories and less commodity/tech exposure than emerging or US markets.

## 6. 2026 Data

Most indexes have only recorded stock data up to February 20th of 2026.

## Limitations of This Analysis

### 1. Local Currency
Each index is measured in its own currency (e.g., BIST100 in TRY, BOVESPA in BRL, SP500 in USD). Cross-index return comparisons conflate equity performance with currency movements. Turkey's +1,470% return is in lira terms — the lira has depreciated massively against USD, so a USD-based investor would see a far lower return.

### 2. Inflation
Nominal returns do not account for changes in local purchasing power. Turkey's extraordinary returns partly reflect hyperinflation-level CPI; real (inflation-adjusted) returns would be substantially lower.

### 3. Dividends
The analysis uses price-only `Close` values. If an index is price-only (not total-return), dividends are excluded. For mature markets like the UK and US where dividend yields can contribute 2–4% annually, the true total-return gap versus emerging markets is narrower than shown.

### 4. Risk Not Reflected in CAGR
CAGR is a single point estimate that does not capture volatility, drawdowns, or recovery time. An index with a higher CAGR may have experienced much larger losses along the way. While this notebook separately computes daily volatility and risk categories, the long-term performance table ranks by CAGR alone without a paired risk metric.

## Why We Cannot Standardize to USD (Yet)

A common-currency (e.g., USD) comparison would make cross-country return rankings more meaningful for a global investor, but the notebook cannot currently perform this conversion:

* **No FX data in the pipeline.** The `silver.index_levels` table contains only `Date`, `Close`, `Daily_Returns_Percentage`, `Index`, `Country`, and `Region` — there is no currency code or exchange-rate column.
* **No FX table exists in the catalog.** Daily TRY/USD, BRL/USD, EUR/USD, and other required rate tables have not been ingested or created in the `global_stock_market` schema.
* **Index-to-currency mapping is missing.** While `Country` is available, there is no lookup table that maps each index to its base currency (e.g., BIST100 → TRY, DAX40 → EUR).

To enable USD-standardized returns, we would need to: (1) source daily FX rates for all 8 non-USD currencies, (2) build a lookup mapping each index to its base currency, (3) join FX rates to `index_levels` on date, and (4) recompute all return metrics on USD-converted closing prices. Until that infrastructure is in place, all returns remain in local currency and should be interpreted with that caveat.